# CFN — Complete Formalism Numerical Audit Template

This notebook template is for **publication-grade numerical audit companions** to Complete Formalism papers.

It is **not** a summary notebook and **not** a loose translation notebook.

Its job is to:

1. mirror the paper in dependency order,
2. turn each load-bearing claim into executable audit logic,
3. declare pass/fail criteria **before** the code that evaluates them,
4. emit at least one **non-generic, claim-specific figure** from each substantive code cell,
5. target any validation gates explicitly stated in the paper,
6. expose residuals, failure modes, and local verdicts at every stage,
7. prove coverage: the notebook must show that it audited the paper rather than sampling it.

Use this template as the starting point for each CF notebook.


## Non-negotiable notebook rules

- **1:1 paper walk-through.** The notebook must mirror the paper's section/subsection order. Compression is not allowed for theorem-bearing material.
- **Gate-before-code discipline.** Every substantive code cell must have a markdown cell immediately above it that declares:
  - the exact claim(s) being audited,
  - the acceptance criteria,
  - the emitted quantities,
  - the emitted figure(s),
  - the failure interpretation.
- **Every substantive code cell must end in a verdict.** Each such cell must produce a structured gate record with a boolean pass/fail field.
- **Every substantive code cell must emit a non-generic figure.** The figure must visualize the exact quantity under audit for that claim. No placeholder sine waves, generic scatter plots, or decorative charts.
- **Paper-gate priority.** If the paper declares validation gates, those gates must appear explicitly in the notebook and must be tested directly.
- **Claim-complete coverage.** The notebook must crawl every load-bearing claim, theorem, definition block, construction step, exclusion, and validation statement that carries burden.
- **Commentary-after-code discipline.** Every substantive code cell must be followed by a markdown commentary cell interpreting the result and stating what passed, what failed, and what remains downstream.
- **Dependency honesty.** No code cell may use objects not yet introduced by the paper stage currently under audit.
- **No theorem laundering.** The notebook may numerically witness or stress-test the paper. It may not silently import missing burden from outside the paper.


## Operating model

The notebook should be treated as an **auditor walking through the paper**.

That means each audited unit should follow this pattern:

1. **Claim unit markdown**
   - Canon anchor
   - Burden type
   - Gate criteria
   - Expected outputs
   - Figure contract

2. **Executable audit cell**
   - deterministic setup
   - translated numerical/constructive implementation
   - metrics
   - verdict dictionary
   - figure(s)

3. **Interpretive commentary markdown**
   - what the result means
   - what would count against the claim
   - what downstream sections are now licensed


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional
import copy
import json
import math
import random
import re

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=10, suppress=True)
SEED = 20260322
random.seed(SEED)
np.random.seed(SEED)

@dataclass
class GateResult:
    gate_id: str
    claim_ids: List[str]
    section_id: str
    burden_type: str
    criteria: Dict[str, Any]
    metrics: Dict[str, Any]
    passed: bool
    figure_paths: List[str]
    figure_description: str
    failure_meaning: str
    notes: str = ""

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

def bool_all(*values: bool) -> bool:
    return all(bool(v) for v in values)

def require_keys(d: Dict[str, Any], keys: List[str], ctx: str) -> List[str]:
    missing = [k for k in keys if k not in d]
    return [f"{ctx}: missing key '{k}'" for k in missing]

def make_gate_result(
    gate_id: str,
    claim_ids: List[str],
    section_id: str,
    burden_type: str,
    criteria: Dict[str, Any],
    metrics: Dict[str, Any],
    passed: bool,
    figure_description: str,
    failure_meaning: str,
    notes: str = "",
) -> GateResult:
    return GateResult(
        gate_id=gate_id,
        claim_ids=claim_ids,
        section_id=section_id,
        burden_type=burden_type,
        criteria=copy.deepcopy(criteria),
        metrics=copy.deepcopy(metrics),
        passed=bool(passed),
        figure_paths=[],
        figure_description=figure_description,
        failure_meaning=failure_meaning,
        notes=notes,
    )

def finalize_figure(title: str, xlabel: str = "", ylabel: str = "") -> None:
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()

def stable_unique(seq: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

def assert_unique(seq: List[str], name: str) -> List[str]:
    seen = set()
    dup = []
    for x in seq:
        if x in seen:
            dup.append(x)
        seen.add(x)
    return [f"duplicate {name}: {x}" for x in dup]

def print_gate(result: GateResult) -> None:
    print(json.dumps(result.to_dict(), indent=2))


## Template metadata contract

Fill these fields for the actual paper before writing the paper-specific audit cells.

- `paper_id`: canonical CF id, e.g. `CF000`
- `paper_title`: exact paper title
- `paper_tex_path`: local path to the LaTeX source when available
- `canonical_markdown`: canonical markdown path or anchor root
- `paper_validation_gates`: the exact gates declared by the paper, if any
- `sections`: the paper's mirrored audit manifest, in dependency order

Each `section` should contain:

- `section_id`
- `paper_label`
- `title`
- `kind` (`front_matter`, `theorem_bearing`, `validation`, `worked_example`, `limits`, `integration`, etc.)
- `claims`: ordered claim units

Each `claim` should contain:

- `claim_id`
- `paper_anchor`
- `title`
- `burden_type`
- `load_bearing` (`True/False`)
- `gate_id`
- `gate_name`
- `criteria`
- `expected_outputs`
- `figure_contract`
- `failure_meaning`
- `depends_on`


In [ ]:
# ------------------------------------------------------------------
# TEMPLATE SPEC
# Replace this object for each paper.
# The tiny example below exists only so the template is runnable.
# ------------------------------------------------------------------

PAPER_SPEC: Dict[str, Any] = {
    "paper_id": "CFx",
    "paper_title": "Complete Formalism Subject Title",
    "paper_tex_path": None,  # e.g. "/absolute/or/relative/path/to/CF000.tex"
    "canonical_markdown": "../../Complete-Formalisms/CFx_{Subject-Title}.md",
    "paper_validation_gates": [
        # Example:
        # {"paper_gate_id": "G1", "title": "No third primitive survivor", "mapped_claim_ids": ["2.3.a", "3.5.a"]},
    ],
    "sections": [
        {
            "section_id": "S0",
            "paper_label": "Template Example",
            "title": "Example claim-unit pattern only",
            "kind": "template_example",
            "claims": [
                {
                    "claim_id": "TEMPLATE.1",
                    "paper_anchor": "#template-example",
                    "title": "Template example gate",
                    "burden_type": "template_example",
                    "load_bearing": False,
                    "gate_id": "TG-EX-001",
                    "gate_name": "Example deterministic numeric audit",
                    "criteria": {
                        "symmetry_residual_max": 1e-12,
                        "grid_size_min": 64
                    },
                    "expected_outputs": [
                        "symmetry_residual",
                        "grid_size",
                        "overall_pass"
                    ],
                    "figure_contract": "Residual field across the audited grid, with the symmetry axis visually identifiable.",
                    "failure_meaning": "The template example is structurally malformed or the helper contract has been broken.",
                    "depends_on": []
                }
            ]
        }
    ]
}


## Gate 0 — Manifest integrity and schema coverage

**Claim being audited:** the notebook manifest is structurally valid enough to support a real paper audit.

**Pass criteria**
- required top-level fields exist,
- section ids are unique and ordered,
- claim ids are unique,
- every claim has gate metadata,
- every load-bearing claim has criteria, expected outputs, figure contract, and dependency list.

**Emitted outputs**
- manifest counts,
- schema error list,
- coverage fractions.

**Figure contract**
- a non-generic manifest coverage figure showing sections vs claim counts and load-bearing claim density.

**Failure meaning**
- the notebook cannot yet be trusted as a paper auditor because coverage bookkeeping is broken before any claim is tested.


In [ ]:
required_top = ["paper_id", "paper_title", "paper_tex_path", "canonical_markdown", "paper_validation_gates", "sections"]
errors = require_keys(PAPER_SPEC, required_top, "PAPER_SPEC")

section_ids = [s.get("section_id", "") for s in PAPER_SPEC.get("sections", [])]
errors += assert_unique(section_ids, "section_id")

all_claim_ids: List[str] = []
load_bearing_claims = 0
claims_missing_gate = 0
claims_missing_criteria = 0
claims_missing_figure = 0
claims_missing_outputs = 0
claims_missing_depends = 0

required_claim_keys = [
    "claim_id", "paper_anchor", "title", "burden_type", "load_bearing",
    "gate_id", "gate_name", "criteria", "expected_outputs",
    "figure_contract", "failure_meaning", "depends_on"
]

section_claim_counts = []
section_load_counts = []

for section in PAPER_SPEC.get("sections", []):
    errors += require_keys(section, ["section_id", "paper_label", "title", "kind", "claims"], f"section:{section.get('section_id','?')}")
    claims = section.get("claims", [])
    section_claim_counts.append(len(claims))
    local_load = 0
    for claim in claims:
        errors += require_keys(claim, required_claim_keys, f"claim:{claim.get('claim_id','?')}")
        cid = claim.get("claim_id", "")
        all_claim_ids.append(cid)
        if bool(claim.get("load_bearing", False)):
            load_bearing_claims += 1
            local_load += 1
        if not claim.get("gate_id"):
            claims_missing_gate += 1
        if not claim.get("criteria"):
            claims_missing_criteria += 1
        if not claim.get("figure_contract"):
            claims_missing_figure += 1
        if not claim.get("expected_outputs"):
            claims_missing_outputs += 1
        if "depends_on" not in claim:
            claims_missing_depends += 1
    section_load_counts.append(local_load)

errors += assert_unique(all_claim_ids, "claim_id")

n_sections = len(PAPER_SPEC.get("sections", []))
n_claims = len(all_claim_ids)
schema_pass = len(errors) == 0
coverage_pass = bool_all(
    claims_missing_gate == 0,
    claims_missing_criteria == 0,
    claims_missing_figure == 0,
    claims_missing_outputs == 0,
    claims_missing_depends == 0,
)

metrics = {
    "n_sections": n_sections,
    "n_claims": n_claims,
    "load_bearing_claims": load_bearing_claims,
    "schema_error_count": len(errors),
    "claims_missing_gate": claims_missing_gate,
    "claims_missing_criteria": claims_missing_criteria,
    "claims_missing_figure": claims_missing_figure,
    "claims_missing_outputs": claims_missing_outputs,
    "claims_missing_depends": claims_missing_depends,
    "schema_errors": errors,
}
criteria = {
    "schema_error_count": 0,
    "claims_missing_gate": 0,
    "claims_missing_criteria": 0,
    "claims_missing_figure": 0,
    "claims_missing_outputs": 0,
    "claims_missing_depends": 0,
}

section_labels = [s["paper_label"] for s in PAPER_SPEC.get("sections", [])] or ["<none>"]
xs = np.arange(len(section_labels))
plt.figure(figsize=(10, 4))
plt.bar(xs - 0.18, section_claim_counts or [0], width=0.35, label="claims")
plt.bar(xs + 0.18, section_load_counts or [0], width=0.35, label="load-bearing")
plt.xticks(xs, section_labels, rotation=30, ha="right")
plt.legend()
finalize_figure(
    title=f"{PAPER_SPEC['paper_id']} manifest coverage by section",
    xlabel="paper section",
    ylabel="claim count",
)
plt.show()

manifest_gate = make_gate_result(
    gate_id="G0-MANIFEST",
    claim_ids=stable_unique(all_claim_ids)[:5] if all_claim_ids else ["<none>"],
    section_id="META",
    burden_type="schema",
    criteria=criteria,
    metrics=metrics,
    passed=bool_all(schema_pass, coverage_pass),
    figure_description="Bar chart of total vs load-bearing claim counts per mirrored paper section.",
    failure_meaning="The notebook manifest is incomplete or malformed, so the paper cannot yet be audited faithfully.",
    notes="This gate audits the notebook specification itself, not the paper's mathematics."
)
print_gate(manifest_gate)


_Commentary (Gate 0):_  
This gate is supposed to fail fast on bookkeeping defects before any mathematical audit begins. In a real paper notebook, this is where you prove that the notebook has a claim manifest rich enough to mirror the paper rather than skipping burden-bearing regions.


## Optional helper — parse section/subsection structure from the paper source

Use this when a LaTeX source file is available. It gives a raw structural skeleton that can be compared against the notebook manifest.

This does **not** prove claim coverage by itself. It only checks whether the notebook's section order can even claim to mirror the paper.


In [ ]:
def parse_latex_headings(tex_text: str) -> List[Dict[str, str]]:
    pattern = re.compile(r"\\(section|subsection|subsubsection)\{([^}]*)\}")
    headings = []
    for m in pattern.finditer(tex_text):
        headings.append({
            "level": m.group(1),
            "title": m.group(2).strip()
        })
    return headings

def load_headings_from_tex(path: Optional[str]) -> List[Dict[str, str]]:
    if path is None:
        return []
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"paper_tex_path does not exist: {p}")
    return parse_latex_headings(p.read_text(encoding="utf-8"))

parsed_headings = load_headings_from_tex(PAPER_SPEC.get("paper_tex_path"))
print(json.dumps(parsed_headings[:20], indent=2))
print(f"Parsed headings: {len(parsed_headings)}")


## Gate 1 — Paper structure mirroring

**Claim being audited:** the notebook's section-level manifest mirrors the paper's section order closely enough to support a true walk-through.

**Pass criteria**
- if `paper_tex_path` is set, every paper section/subsection heading is either mirrored directly or explicitly justified as non-load-bearing front matter,
- notebook section order is monotone with paper order,
- notebook does not invent theorem-bearing sections absent from the paper.

**Emitted outputs**
- parsed heading count,
- mirrored heading count,
- unmatched heading count.

**Figure contract**
- a section-order alignment figure comparing paper heading index to notebook section index.

**Failure meaning**
- the notebook cannot yet claim 1:1 structural fidelity to the paper.


In [ ]:
paper_titles = [h["title"] for h in parsed_headings if h["level"] in {"section", "subsection"}]
notebook_titles = [s["title"] for s in PAPER_SPEC.get("sections", [])]

matched = []
unmatched_paper = []
for i, t in enumerate(paper_titles):
    if t in notebook_titles:
        matched.append((i, notebook_titles.index(t)))
    else:
        unmatched_paper.append(t)

unmatched_notebook = [t for t in notebook_titles if t not in paper_titles] if paper_titles else []

criteria = {
    "unmatched_paper_heading_count": 0 if paper_titles else None,
    "unmatched_notebook_heading_count": 0 if paper_titles else None,
    "monotone_index_alignment": True if paper_titles else None,
}
monotone = True
if matched:
    y = [j for _, j in matched]
    monotone = all(y[i] <= y[i+1] for i in range(len(y)-1))
else:
    y = []

metrics = {
    "parsed_heading_count": len(paper_titles),
    "notebook_section_count": len(notebook_titles),
    "matched_heading_count": len(matched),
    "unmatched_paper_heading_count": len(unmatched_paper),
    "unmatched_notebook_heading_count": len(unmatched_notebook),
    "unmatched_paper_titles": unmatched_paper[:20],
    "unmatched_notebook_titles": unmatched_notebook[:20],
    "monotone_index_alignment": monotone,
}

plt.figure(figsize=(6, 6))
if matched:
    x = [i for i, _ in matched]
    y = [j for _, j in matched]
    plt.plot(x, y, marker="o")
    lim = max(max(x), max(y)) + 1
    plt.plot([0, lim], [0, lim], linestyle="--")
else:
    plt.scatter([0], [0])
finalize_figure(
    title=f"{PAPER_SPEC['paper_id']} paper/notebook section alignment",
    xlabel="paper heading index",
    ylabel="notebook section index",
)
plt.show()

passed = True if not paper_titles else bool_all(
    len(unmatched_paper) == 0,
    len(unmatched_notebook) == 0,
    monotone,
)

structure_gate = make_gate_result(
    gate_id="G1-STRUCTURE",
    claim_ids=[],
    section_id="META",
    burden_type="structure_alignment",
    criteria=criteria,
    metrics=metrics,
    passed=passed,
    figure_description="Index-alignment plot between parsed paper headings and notebook mirrored sections.",
    failure_meaning="The notebook does not yet mirror the paper's order closely enough to count as a true audit walk-through.",
    notes="When paper_tex_path is unset, this gate acts as a soft scaffold and should be tightened during instantiation."
)
print_gate(structure_gate)


_Commentary (Gate 1):_  
For a real CF notebook, this gate should become strict once the paper source is attached. A mismatched or compressed section plan is already evidence that the notebook is sampling the paper instead of crawling it.


## Gate 2 — Paper-declared validation gate coverage

**Claim being audited:** every validation gate named by the paper has at least one explicit notebook audit target.

**Pass criteria**
- every `paper_validation_gates[*].paper_gate_id` has at least one mapped claim id,
- every mapped claim id exists in the manifest,
- no paper-level validation gate is left orphaned.

**Emitted outputs**
- gate counts,
- orphan counts,
- bad mapping counts.

**Figure contract**
- a gate coverage chart showing paper gates and the number of notebook claim units attached to each.

**Failure meaning**
- the notebook has not actually targeted the paper's own validation logic.


In [ ]:
claim_id_set = set(all_claim_ids)
paper_gates = PAPER_SPEC.get("paper_validation_gates", [])
gate_labels = []
gate_counts = []
orphans = 0
bad_maps = 0
details = []

for g in paper_gates:
    pgid = g.get("paper_gate_id", "<missing>")
    mapped = g.get("mapped_claim_ids", [])
    gate_labels.append(pgid)
    gate_counts.append(len(mapped))
    if len(mapped) == 0:
        orphans += 1
    bad = [cid for cid in mapped if cid not in claim_id_set]
    if bad:
        bad_maps += len(bad)
    details.append({"paper_gate_id": pgid, "mapped_claim_ids": mapped, "bad_claim_ids": bad})

criteria = {
    "orphan_paper_gate_count": 0,
    "bad_mapped_claim_id_count": 0,
}
metrics = {
    "paper_gate_count": len(paper_gates),
    "orphan_paper_gate_count": orphans,
    "bad_mapped_claim_id_count": bad_maps,
    "details": details,
}

plt.figure(figsize=(10, 4))
if gate_labels:
    xs = np.arange(len(gate_labels))
    plt.bar(xs, gate_counts)
    plt.xticks(xs, gate_labels, rotation=30, ha="right")
else:
    plt.bar([0], [0])
    plt.xticks([0], ["<no paper gates listed>"])
finalize_figure(
    title=f"{PAPER_SPEC['paper_id']} paper-declared gate coverage",
    xlabel="paper gate id",
    ylabel="mapped notebook claim count",
)
plt.show()

paper_gate_cov = make_gate_result(
    gate_id="G2-PAPER-GATES",
    claim_ids=[],
    section_id="META",
    burden_type="validation_gate_coverage",
    criteria=criteria,
    metrics=metrics,
    passed=bool_all(orphans == 0, bad_maps == 0),
    figure_description="Bar chart of paper-declared validation gates vs number of mapped notebook claim units.",
    failure_meaning="The notebook fails to target the validation gates declared by the paper itself.",
    notes="If the paper declares no explicit gates, this should be replaced with derived local gates during paper instantiation."
)
print_gate(paper_gate_cov)


_Commentary (Gate 2):_  
This is where the notebook proves it is pointed at the paper's own validation burden rather than inventing a separate success criterion.


## Example audited claim unit pattern

The following pair is an **example only**. Replace it with the first real claim unit from the target paper.


### Example claim unit

**Canon anchor:** `#template-example`  
**Burden type:** template example  
**Claim under audit:** a symmetric constructed field should have reflection residual below tolerance.

**Pass criteria**
- maximum reflection residual ≤ `1e-12`,
- grid size ≥ `64`.

**Emitted outputs**
- reflection residual field,
- max residual,
- grid size,
- pass/fail verdict.

**Figure contract**
- heat map of the pointwise reflection residual across the computational domain.

**Failure meaning**
- the implemented construction fails its own symmetry condition or the gate logic is wrong.


In [ ]:
# Example deterministic audit cell — replace with a paper-specific implementation.
x = np.linspace(-1.0, 1.0, 129)
X, Y = np.meshgrid(x, x)
Z = X**2 + 0.5 * Y**2  # exactly even under X -> -X
Z_reflect = np.flip(Z, axis=1)
residual = np.abs(Z - Z_reflect)

max_residual = float(np.max(residual))
grid_size = int(Z.shape[0])
criteria = {
    "max_reflection_residual": 1e-12,
    "grid_size_min": 64,
}
metrics = {
    "max_reflection_residual": max_residual,
    "grid_size": grid_size,
    "overall_pass": bool_all(max_residual <= criteria["max_reflection_residual"], grid_size >= criteria["grid_size_min"]),
}

plt.figure(figsize=(6, 5))
plt.imshow(residual, origin="lower", extent=[x.min(), x.max(), x.min(), x.max()])
plt.colorbar(label="|Z(x,y) - Z(-x,y)|")
finalize_figure(
    title="Template example — reflection residual field",
    xlabel="x",
    ylabel="y",
)
plt.show()

template_example_gate = make_gate_result(
    gate_id="TG-EX-001",
    claim_ids=["TEMPLATE.1"],
    section_id="S0",
    burden_type="template_example",
    criteria=criteria,
    metrics=metrics,
    passed=metrics["overall_pass"],
    figure_description="Pointwise residual heat map comparing the field to its reflected counterpart.",
    failure_meaning="The example construction is not satisfying the declared reflection gate.",
)
print_gate(template_example_gate)


_Commentary (Example):_  
This is the pattern to reuse. The important part is not the toy symmetry check. The important part is the sequence: claim declaration → explicit criteria → deterministic execution → specific figure → structured verdict → commentary.


## Section-level expansion pattern for real papers

For the instantiated notebook, repeat the following trio for **every load-bearing paper unit**:

1. Markdown gate declaration for Section X.Y / Theorem / Definition / Exclusion / Construction step  
2. Code cell that computes the audit quantities and emits a gate record + specific figure  
3. Markdown commentary cell

A paper with 25 load-bearing subsections should therefore usually produce roughly 75 local cells, not 5 compressed blocks.


## Publication bundle expectations

A publishable CF notebook should include:

- exact paper id and title,
- canonical anchor path(s),
- mirrored paper manifest,
- paper-gate coverage report,
- per-claim local gate results,
- section-level commentary,
- deterministic run header,
- final notebook-wide audit summary.

It should **not** hide missing burden behind:
- 'illustrative only'
- 'qualitative check'
- 'placeholder'
- 'future work'  
inside theorem-bearing regions.


## Final instantiation checklist

Before publishing a paper-specific notebook built from this template, confirm:

- [ ] paper source attached or section manifest manually verified
- [ ] paper headings mirrored in order
- [ ] every load-bearing claim given a local gate
- [ ] every substantive code cell preceded by criteria markdown
- [ ] every substantive code cell emits a non-generic figure
- [ ] every paper-declared validation gate targeted
- [ ] every code result followed by commentary
- [ ] final summary aggregates all local verdicts
- [ ] no hidden imports of later theory or external burden
